<a href="https://colab.research.google.com/github/Varsanrecp/ML-portfolio/blob/main/Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**LANGCHAIN PDF FILE Q/A CHATBOT**

In [1]:
# Run this cell first in Colab
!pip install -qU langchain langchain-google-genai google-genai chromadb pypdf tiktoken


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.7/244.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 86.3 MB/s eta 

In [8]:
pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
import os, getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google / Gemini API key (from Google AI Studio): ")

# set GEMINI_API_KEY too for compatibility with some examples
os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

print("API key loaded into environment variables (not printed).")


Enter your Google / Gemini API key (from Google AI Studio): ··········
API key loaded into environment variables (not printed).


In [5]:
# Option A: Upload from your machine
from google.colab import files
uploaded = files.upload()  # select the PDF file(s)
pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)



Saving Instructions Python Developer (1).pdf to Instructions Python Developer (1).pdf
Uploaded: Instructions Python Developer (1).pdf


In [9]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
pages = loader.load()                       # loads as LangChain Document objects (one per page)
print(f"Loaded {len(pages)} page-documents from PDF.")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(pages)      # list of Document objects (chunks)
print(f"Split into {len(docs)} chunks. Example chunk length:", len(docs[0].page_content))


Loaded 3 page-documents from PDF.
Split into 5 chunks. Example chunk length: 994


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import Chroma

# create embeddings client (will use GOOGLE_API_KEY from env if not passed)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# build Chroma vector store from documents
persist_dir = "./chroma_db"   # change if you want
vectordb = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=persist_dir)
vectordb.persist()  # optional but handy to reuse later
print("Vector DB created & persisted at", persist_dir)


Vector DB created & persisted at ./chroma_db


/tmp/ipython-input-1249957150.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()  # optional but handy to reuse later


In [11]:
from langchain_google_genai import GoogleGenerativeAI
from langchain.chains import RetrievalQA

# LLM (Gemini)
llm = GoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.0)  # deterministic answers

# retriever (k = how many chunks to fetch)
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# make a QA chain (returns answer + optionally source documents)
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)
print("RetrievalQA ready.")


RetrievalQA ready.


In [12]:
def ask(question: str):
    out = qa({"query": question})
    # output shape differs by versions; common keys: "result" (text) and "source_documents"
    answer = out.get("result") or out.get("answer") or out
    print("\n=== Answer ===\n")
    print(answer)
    print("\n=== Top source chunks (for attribution) ===\n")
    for i, doc in enumerate(out.get("source_documents", [])[:3]):
        print(f"--- source {i+1} ---")
        print("source:", doc.metadata.get("source", "unknown"))
        print(doc.page_content[:400].strip(), "\n")

# Try it:
ask("Summarize the document in two lines.")


/tmp/ipython-input-926637681.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  out = qa({"query": question})



=== Answer ===

This document is an assignment to develop a command-line trading bot for Binance Futures that supports core and advanced order types. The project requires robust logging, clear documentation, and submission with a specific file structure.

=== Top source chunks (for attribution) ===

--- source 1 ---
source: Instructions Python Developer (1).pdf
Key Changes from OriginalNo code snippets: Only conceptual examples (e.g., "OCO orders").Flexible naming: Emphasized descriptive filenames over hardcoded ones.GitHub integration: Added private repo instructions.Prioritization clarity: Highlighted advanced orders as a key differentiator. 

--- source 2 ---
source: Instructions Python Developer (1).pdf
Guidelines1. File StructureSubmit a single `.zip` file named `[your_name]_binance_bot.zip` with this structure:[project_root]/│├── /src/ # All source code│├── market_orders.py # Example: Market order logic│├── limit_orders.py # Example: Limit order logic│├── advanced/ # (Bonus)

In [13]:
def ask(question: str):
    out = qa({"query": question})
    # output shape differs by versions; common keys: "result" (text) and "source_documents"
    answer = out.get("result") or out.get("answer") or out
    print("\n=== Answer ===\n")
    print(answer)
    # print("\n=== Top source chunks (for attribution) ===\n")
    # for i, doc in enumerate(out.get("source_documents", [])[:3]):
    #     print(f"--- source {i+1} ---")
    #     print("source:", doc.metadata.get("source", "unknown"))
    #     print(doc.page_content[:400].strip(), "\n")

# Try it:
ask("Summarize the document in two lines.")



=== Answer ===

This document outlines an assignment to build a command-line trading bot for Binance Futures that supports various order types. The project requires robust logging, clear documentation, and submission in a specific `.zip` file structure.
